Elisa Aguirre Arias

2/19/26

738894

# Regresión - Motor Trend Car Road Tests


## 1.1 Regresión Lineal con `mpg` como variable objetivo

In [14]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge

In [7]:
df = pd.read_excel(r"C:\Users\elisa\Downloads\Motor Trend Car Road Tests.xlsx")

### Ajuste del modelo con todos los datos y interpretación de los coeficientes (Betas)
Se ajusta un modelo de regresión lineal utilizando todos los datos disponibles.
Se calcula el coeficiente de determinación $R^2$.
Los signos de los coeficientes indican:

- Beta positivo → relación directa con `mpg`
- Beta negativo → relación inversa con `mpg`

In [8]:
df = df.drop(columns=["model"])

In [9]:
X = df.drop(columns=["mpg"])
y = df["mpg"]

In [10]:
modelo = LinearRegression()
modelo.fit(X, y)
r2 = modelo.score(X, y)
print("R2:", r2)

R2: 0.8690157644777646


In [11]:
betas = pd.Series(modelo.coef_, index=X.columns)
print(betas)

cyl    -0.111440
disp    0.013335
hp     -0.021482
drat    0.787111
wt     -3.715304
qsec    0.821041
vs      0.317763
am      2.520227
gear    0.655413
carb   -0.199419
dtype: float64


### Interpretación de los coeficientes (Betas)

Los coeficientes estimados muestran cómo cambia el `mpg` cuando cada variable aumenta en una unidad, manteniendo las demás constantes.

- **cyl (-0.111)**  
  A mayor número de cilindros, menor rendimiento de gasolina. Esto tiene sentido ya que motores con más cilindros suelen consumir más combustible.

- **disp (0.013)**  
  El desplazamiento del motor tiene un efecto positivo muy pequeño sobre el mpg, por lo que su impacto es mínimo en comparación con otras variables.

- **hp (-0.021)**  
  A mayor caballaje, menor mpg. Los autos más potentes tienden a consumir más gasolina.

- **drat (0.787)**  
  Una mayor relación del eje diferencial se asocia con un mayor rendimiento de gasolina.

- **wt (-3.715)**  
  Es el coeficiente de mayor magnitud negativa. Indica que el peso del vehículo tiene un fuerte impacto negativo sobre el mpg. Autos más pesados consumen significativamente más combustible.

- **qsec (0.821)**  
  Un mayor tiempo en el cuarto de milla (autos menos rápidos) se asocia con mayor rendimiento de gasolina.

- **vs (0.318)**  
  Los motores en línea (vs = 1) tienden a tener un mayor mpg en comparación con motores en V.

- **am (2.520)**  
  Los autos con transmisión manual presentan mayor rendimiento de gasolina que los automáticos.

- **gear (0.655)**  
  Un mayor número de velocidades se asocia con un incremento en el mpg.

- **carb (-0.199)**  
  Más carburadores se relacionan con menor rendimiento de gasolina.

En general, las variables que más influyen en el modelo por su magnitud son `wt`, `am`, `qsec` y `drat`. Especialmente el peso (`wt`) es el factor que más afecta negativamente el rendimiento de gasolina.


### División Train-Test (40% entrenamiento)

Se divide el dataset usando:

- 40% para entrenamiento
- 60% para prueba

Se comparan los valores de $R^2$ en entrenamiento y prueba para evaluar posible sobreajuste.


In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    train_size=0.40,
    random_state=42)
modelo = LinearRegression()
modelo.fit(X_train, y_train)
print("R2 entrenamiento:", modelo.score(X_train, y_train))
print("R2 prueba:", modelo.score(X_test, y_test))

R2 entrenamiento: 0.9982122857062323
R2 prueba: -7.10714065351071


###  Regularización L2 (Ridge Regression)

Se añade regularización L2 para penalizar coeficientes grandes y reducir posible sobreajuste.

Se prueba con distintos valores de $\lambda$ (alpha).


In [17]:
ridge = Ridge(alpha=1) 
ridge.fit(X_train, y_train)
print("R2 entrenamiento (Ridge):", ridge.score(X_train, y_train))
print("R2 prueba (Ridge):", ridge.score(X_test, y_test))

R2 entrenamiento (Ridge): 0.9278776695116555
R2 prueba (Ridge): 0.631079320358904


In [16]:
for l in [0.01, 0.1, 1, 10, 100]:
    ridge = Ridge(alpha=l)
    ridge.fit(X_train, y_train)
    print("Lambda:", l)
    print("R2 train:", ridge.score(X_train, y_train))
    print("R2 test:", ridge.score(X_test, y_test))
    print("------")


Lambda: 0.01
R2 train: 0.9951427860865372
R2 test: -2.3330784469876256
------
Lambda: 0.1
R2 train: 0.9794200798509612
R2 test: 0.26067024754956536
------
Lambda: 1
R2 train: 0.9278776695116555
R2 test: 0.631079320358904
------
Lambda: 10
R2 train: 0.8633880241271136
R2 test: 0.6567019278519768
------
Lambda: 100
R2 train: 0.8073152721086998
R2 test: 0.599017961383292
------


### Análisis de la regularización L2 (Ridge)

Se probaron distintos valores de $\lambda$ para analizar cómo cambia el desempeño del modelo.

| Lambda | R2 Train | R2 Test |
|--------|----------|----------|
| 0.01   | 0.995    | -2.333   |
| 0.1    | 0.979    | 0.261    |
| 1      | 0.928    | 0.631    |
| 10     | 0.864    | 0.657    |
| 100    | 0.807    | 0.599    |

### Interpretación

- Con **λ muy pequeño (0.01)** el modelo se parece mucho a la regresión normal.  
  El R2 de entrenamiento es extremadamente alto (0.995), pero el R2 de prueba es negativo (-2.33), lo que indica un fuerte sobreajuste.

- Conforme aumenta λ, el modelo se vuelve más estable.  
  El R2 de entrenamiento disminuye, pero el R2 de prueba mejora considerablemente.

- El mejor desempeño en prueba se obtiene aproximadamente con **λ = 10**, donde:
  - R2 train ≈ 0.864
  - R2 test ≈ 0.657

- Cuando λ es demasiado grande (100), el modelo empieza a perder capacidad predictiva, ya que la penalización es excesiva.

### Conclusión

La regularización L2 ayuda a reducir el sobreajuste.  
Un valor intermedio de λ (como 10) logra un mejor balance entre ajuste y capacidad de generalización.


## 1.2 Regresión Lineal con `qsec` como variable objetivo

En este ejercicio se repite el procedimiento anterior, pero ahora la variable de salida es `qsec`.

Se elimina la columna `model` y todos los demás factores se consideran numéricos.


In [18]:
X = df.drop(columns=["qsec"])
y = df["qsec"]


In [19]:
modelo = LinearRegression()
modelo.fit(X, y)
print("R2 total:", modelo.score(X, y))
betas = pd.Series(modelo.coef_, index=X.columns)
print(betas)

R2 total: 0.8746925777093999
mpg     0.069048
cyl    -0.362678
disp   -0.007501
hp     -0.001563
drat   -0.131064
wt      1.496332
vs      0.970035
am     -0.901186
gear   -0.201285
carb   -0.273598
dtype: float64


### Interpretación de los coeficientes (`qsec`)

Los coeficientes indican cómo cambia el tiempo en el cuarto de milla (`qsec`) cuando cada variable aumenta en una unidad, manteniendo las demás constantes.

- **mpg (0.069)**  
  Mayor rendimiento de gasolina se asocia con un ligero aumento en el tiempo del cuarto de milla.

- **cyl (-0.363)**  
  Más cilindros reducen el tiempo, es decir, autos más potentes son más rápidos.

- **disp (-0.007)**  
  Mayor desplazamiento reduce ligeramente el tiempo.

- **hp (-0.0015)**  
  Más caballos de fuerza reducen el tiempo del cuarto de milla.

- **drat (-0.131)**  
  Mayor relación diferencial reduce el tiempo.

- **wt (1.496)**  
  Autos más pesados tardan más en recorrer el cuarto de milla.

- **vs (0.970)**  
  Motores en línea tienden a ser ligeramente más lentos.

- **am (-0.901)**  
  Transmisión manual reduce el tiempo, es decir, mejora aceleración.

- **gear (-0.201)**  
  Más velocidades reducen el tiempo.

- **carb (-0.274)**  
  Más carburadores reducen el tiempo.


In [21]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    train_size=0.40,
    random_state=42)
modelo.fit(X_train, y_train)
print("R2 entrenamiento:", modelo.score(X_train, y_train))
print("R2 prueba:", modelo.score(X_test, y_test))

R2 entrenamiento: 0.9989478049029176
R2 prueba: -1.0013422169956154


In [22]:
for l in [0.01, 0.1, 1, 10, 100]:
    ridge = Ridge(alpha=l)
    ridge.fit(X_train, y_train)
    print("Lambda:", l)
    print("R2 train:", ridge.score(X_train, y_train))
    print("R2 test:", ridge.score(X_test, y_test))
    print("------")

Lambda: 0.01
R2 train: 0.9975150251299634
R2 test: -0.11676324538500316
------
Lambda: 0.1
R2 train: 0.9855785599163909
R2 test: 0.6878327440279581
------
Lambda: 1
R2 train: 0.934724387000222
R2 test: 0.7141258477214347
------
Lambda: 10
R2 train: 0.8453882031592018
R2 test: 0.5044449278883252
------
Lambda: 100
R2 train: 0.7836618000843937
R2 test: 0.423188008290161
------


### Análisis de la regularización L2 para `qsec`

Se probaron distintos valores de $\lambda$ para analizar cómo cambia el desempeño del modelo.

| Lambda | R2 Train | R2 Test |
|--------|----------|----------|
| 0.01   | 0.998    | -0.117   |
| 0.1    | 0.986    | 0.688    |
| 1      | 0.935    | 0.714    |
| 10     | 0.845    | 0.504    |
| 100    | 0.784    | 0.423    |

### Interpretación

- Con **λ = 0.01**, el modelo sigue sobreajustado.  
  El R2 de entrenamiento es casi perfecto (0.998), pero el R2 de prueba es negativo.

- Con **λ = 0.1**, el modelo mejora considerablemente en prueba (R2 ≈ 0.69).

- El mejor desempeño en prueba se obtiene con **λ = 1**, donde:
  - R2 train ≈ 0.935
  - R2 test ≈ 0.714

- Cuando λ es demasiado grande (10 o 100), el modelo comienza a perder capacidad predictiva, ya que la penalización es excesiva.

### Conclusión

La regularización L2 reduce significativamente el sobreajuste observado en el modelo original.

Un valor intermedio de λ (alrededor de 1) logra el mejor balance entre ajuste y generalización.


## 2.1 Regresión Lineal con `mpg` usando variables dummy

En este ejercicio se realiza una regresión tomando `mpg` como variable objetivo.

Se elimina la columna `model` y se crean variables dummy para los factores:
- `cyl`
- `gear`
- `carb`

El objetivo es analizar si tratar estas variables como categóricas mejora el desempeño del modelo.


In [30]:
df2 = pd.read_excel(r"C:\Users\elisa\Downloads\Motor Trend Car Road Tests.xlsx")
df2 = df2.drop(columns=["model"])
df2 = pd.get_dummies(df2, columns=["cyl", "gear", "carb"], drop_first=True)


In [31]:
X = df2.drop(columns=["mpg"])
y = df2["mpg"]

**Ajuste del modelo con todos los datos**

Se ajusta una regresión lineal con las variables dummy.
Se calcula el coeficiente de determinación $R^2$.


In [32]:
modelo = LinearRegression()
modelo.fit(X, y)
print("R2 total:", modelo.score(X, y))
betas = pd.Series(modelo.coef_, index=X.columns)
print(betas)

R2 total: 0.8930749320864843
disp      0.035546
hp       -0.070507
drat      1.182830
wt       -4.529776
qsec      0.367845
vs        1.930851
am        1.212116
cyl_6    -2.648695
cyl_8    -0.336163
gear_4    1.114355
gear_5    2.528396
carb_2   -0.979354
carb_3    2.999639
carb_4    1.091423
carb_6    4.477569
carb_8    7.250411
dtype: float64


### Interpretación de los coeficientes (modelo con dummies)

Las variables continuas mantienen su interpretación habitual:

- **wt (-4.53)**  
  El peso es el factor más influyente. A mayor peso, menor rendimiento de gasolina.

- **hp (-0.07)**  
  Mayor potencia reduce el mpg.

- **drat (1.18)**  
  Mayor relación diferencial aumenta el mpg.

- **am (1.21)**  
  Transmisión manual aumenta el rendimiento de gasolina.



Las variables dummy se interpretan respecto a su categoría base:

#### Cilindros (categoría base: 4 cilindros)

- **cyl_6 (-2.65)**  
  Los autos de 6 cilindros tienen aproximadamente 2.65 mpg menos que los de 4 cilindros.

- **cyl_8 (-0.34)**  
  Los autos de 8 cilindros tienen ligeramente menor mpg que los de 4 cilindros.



#### Gear (categoría base: 3 velocidades)

- **gear_4 (1.11)**  
  Autos con 4 velocidades tienen mayor mpg que los de 3 velocidades.

- **gear_5 (2.53)**  
  Autos con 5 velocidades tienen aún mayor mpg comparado con la base.



#### Carb (categoría base: 1 carburador)

- **carb_2 (-0.98)**  
  Dos carburadores reducen el mpg respecto a la base.

- **carb_3 (2.99)**  
  Tres carburadores aumentan el mpg.

- **carb_4 (1.09)**  
  Cuatro carburadores aumentan ligeramente el mpg.

- **carb_6 (4.48)**  
  Se observa un aumento considerable del mpg.

- **carb_8 (7.25)**  
  Presenta el mayor incremento respecto a la categoría base.


**División Train-Test (40% entrenamiento)**

Se divide el dataset en:
- 40% entrenamiento
- 60% prueba

Se comparan los valores de $R^2$.


In [33]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    train_size=0.40,
    random_state=42)
modelo = LinearRegression()
modelo.fit(X_train, y_train)
print("R2 entrenamiento:", modelo.score(X_train, y_train))
print("R2 prueba:", modelo.score(X_test, y_test))


R2 entrenamiento: 1.0
R2 prueba: -1.3252877613458587


## 2.2 Regresión Lineal con `qsec` usando variables dummy

En este ejercicio se realiza una regresión tomando `qsec` como variable objetivo.

Se elimina la columna `model` y se crean variables dummy para:
- `cyl`
- `gear`
- `carb`

Se analiza el desempeño del modelo y se interpretan los coeficientes.


In [35]:
df3 = pd.read_excel(r"C:\Users\elisa\Downloads\Motor Trend Car Road Tests.xlsx")
df3 = df3.drop(columns=["model"])
df3 = pd.get_dummies(df3, columns=["cyl", "gear", "carb"], drop_first=True)

In [36]:
X = df3.drop(columns=["qsec"])
y = df3["qsec"]

**Ajuste del modelo con todos los datos**


In [37]:
modelo = LinearRegression()
modelo.fit(X, y)
print("R2 total:", modelo.score(X, y))
betas = pd.Series(modelo.coef_, index=X.columns)
print(betas)


R2 total: 0.9082689440167556
mpg       0.027741
disp      0.003531
hp       -0.002066
drat      0.107866
wt        0.810472
vs        0.265732
am       -1.694294
cyl_6    -1.104343
cyl_8    -2.966901
gear_4    1.332282
gear_5    0.182317
carb_2   -0.834413
carb_3   -0.229304
carb_4   -1.947034
carb_6   -1.565241
carb_8   -1.332317
dtype: float64


### Interpretación de los coeficientes (`qsec` con dummies)

Las variables continuas indican cómo cambia el tiempo en el cuarto de milla cuando la variable aumenta una unidad.

- **wt (0.81)**  
  Autos más pesados tardan más tiempo en recorrer el cuarto de milla.

- **hp (-0.002)**  
  Mayor potencia reduce ligeramente el tiempo.

- **am (-1.69)**  
  Transmisión manual reduce el tiempo, es decir, mejora aceleración.


Las variables dummy se interpretan respecto a su categoría base.

#### Cilindros (base: 4 cilindros)

- **cyl_6 (-1.10)**  
  Autos de 6 cilindros reducen el tiempo respecto a 4 cilindros.

- **cyl_8 (-2.97)**  
  Autos de 8 cilindros reducen aún más el tiempo, indicando mayor velocidad.



#### Gear (base: 3 velocidades)

- **gear_4 (1.35)**  
  Autos con 4 velocidades tardan más que los de 3.

- **gear_5 (0.18)**  
  El efecto es pequeño respecto a la base.


#### Carb (base: 1 carburador)

- Coeficientes negativos indican reducción del tiempo (mayor aceleración).
- Coeficientes positivos indican aumento del tiempo.

En general, más cilindros y transmisión manual reducen el tiempo del cuarto de milla.


**División Train-Test (40% entrenamiento)**


In [38]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    train_size=0.40,
    random_state=42)
modelo = LinearRegression()
modelo.fit(X_train, y_train)
print("R2 entrenamiento:", modelo.score(X_train, y_train))
print("R2 prueba:", modelo.score(X_test, y_test))

R2 entrenamiento: 1.0
R2 prueba: -0.05999228859335193


## 3.1 Comparación entre 1.1 y 2.1 (`mpg`)

Se comparan los modelos:

- 1.1: Variables numéricas
- 2.1: Variables dummy para `cyl`, `gear` y `carb`

### Resultados

- 1.1 R2 total ≈ 0.87  
- 2.1 R2 total ≈ 0.89  

Aunque el modelo con dummies presenta un R2 total ligeramente mayor, al analizar el desempeño en prueba se observa:

- 1.1 R2 test (con regularización) ≈ 0.63
- 2.1 R2 test ≈ -1.33

### Conclusión

El modelo con dummies presenta mayor sobreajuste debido al incremento en el número de parámetros.

El modelo 1.1 con regularización L2 ofrece mejor capacidad de generalización y desempeño en datos de prueba.

Por lo tanto, para `mpg`, tratar las variables como categóricas no mejora el modelo en términos de generalización.


## 3.2 Comparación entre 1.2 y 2.2 (`qsec`)

Se comparan los modelos:

- 1.2: Variables numéricas
- 2.2: Variables dummy para `cyl`, `gear` y `carb`

### Resultados en prueba

- 1.2 sin regularización → R2 ≈ -1.00  
- 1.2 con regularización (λ ≈ 1) → R2 ≈ 0.71  
- 2.2 con dummies → R2 ≈ -0.06  

### Análisis

El modelo 2.2 mejora ligeramente respecto al modelo sin regularización, pero sigue mostrando problemas de sobreajuste.

El mejor desempeño se obtiene en el modelo 1.2 cuando se aplica regularización L2, logrando un R2 en prueba cercano a 0.71.

### Conclusión

Para `qsec`, agregar variables dummy no mejora el desempeño del modelo.

La regularización L2 es más efectiva para mejorar la capacidad de generalización que aumentar la complejidad del modelo mediante variables categóricas.
